### Filter Verification V1

In [ ]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import pywt
import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

from scipy.stats import spearmanr
from scipy.stats import pearsonr
from scipy.stats import linregress
from scipy.stats import fisher_exact
from scipy.signal import butter
from scipy.signal import lfilter
from sklearn.metrics import roc_auc_score
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.filters.hp_filter import hpfilter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score
)
from sklearn.feature_selection import (
    mutual_info_regression
)
from pykalman import KalmanFilter

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.request_api_v2 import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension
from Modules.fibonacci_retracement import FibonacciRetracement
from Modules.fibonacci_extension import FibonacciExtension

In [2]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
fred_url = "https://api.stlouisfed.org/fred/series/observations"
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()
# Fibonacciエクステンションをプロットするクラスのインスタンスを作成
fibonacci_extension_v2 = FibonacciExtension() 
# Fibonacciリトレースメントを計算するクラスのインスタンスを作成
fibonacci_retracement = FibonacciRetracement()

In [3]:
# Date range for fetching API s
start = '1950-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
frequency = "d"
aggregation_method = "avg"

In [4]:
def get_fred_timeseries_data(
    observation_start: str,
    observation_end: str,
    series_mapping: dict[str, str] | None = None,
    frequency: str | None = None,
    aggregation_method: str | None = None,
) -> dict[str, pd.DataFrame]:
    observation_start = observation_start
    observation_end = observation_end
    base_fred_params = {
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": observation_end,
        "frequency": frequency or "m",
        "aggregation_method": aggregation_method or "avg",
    }
    fred_dfs = {}
    for series_id, series_name in series_mapping.items():
        fred_params = base_fred_params.copy()
        fred_params["series_id"] = series_id
        fred_res = requests.get(fred_url, params=fred_params)
        fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
            "date": lambda df: pd.to_datetime(df["date"]),
            "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
            "interest_rate_type": series_name,
        })
        fred_dfs[series_name] = fred_df
        time.sleep(5)
        
    return fred_dfs

In [5]:
def delete_column(df):
    df = df.drop(columns=['realtime_start', 'realtime_end', 'interest_rate_type'])
    return df

#### S&P500(均等加重)時系列を取得

In [6]:
spxew_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^SPXEW",
    market=None,
    series_name="SPXEW"
))
spxew_df.tail()

取得件数: 4989


,value
date,
2026-10-23,8841.269531
2026-10-26,8772.370117
2026-10-27,8802.639648
2026-10-28,8809.419922
2026-10-29,8806.370117


#### S&P500(時価総額加重平均)時系列を取得

In [7]:
sp500_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^GSPC",
    market=None,
    series_name="GSPC"
))
sp500_df.tail()

取得件数: 19359


,value
date,
2026-10-23,7637.180176
2026-10-26,7639.700195
2026-10-27,7699.470215
2026-10-28,7735.560059
2026-10-29,7699.609863


#### Nasdaq100(均等加重)時系列を取得

In [8]:
ndxe_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^NDXE",
    market=None,
    series_name="NDXE"
))
ndxe_df.tail()

取得件数: 5150


,value
date,
2026-10-23,9893.769531
2026-10-26,9917.309570
2026-10-27,10010.910156
2026-10-28,10064.759766
2026-10-29,10038.530273


#### Nasdaq100(時価総額加重平均)時系列を取得

In [9]:
qqq_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="^IXIC",
    market=None,
    series_name="IXIC"
))
qqq_df.tail()

取得件数: 14051


,value
date,
2026-10-23,25893.949219
2026-10-26,26026.869141
2026-10-27,26286.310547
2026-10-28,26541.609375
2026-10-29,26201.820312


#### iShares iBoxx $ High Yield Corporate Bond ETF

In [10]:
hyg_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="HYG",
    market=None,
    series_name="HYG"
))
hyg_df.tail()

取得件数: 4921


,value
date,
2026-10-23,76.889999
2026-10-26,77.080002
2026-10-27,77.300003
2026-10-28,77.709999
2026-10-29,77.480003


#### iShares 7-10 Year Treasury Bond ETF (IEF)

In [11]:
ief_df = delete_column(request_api.fetch_stock_timeseries_data_fred_format(
    start_date=start,
    end_date=end,
    code="IEF",
    market=None,
    series_name="IEF"
))
ief_df.tail()

取得件数: 6103


,value
date,
2026-10-23,91.230003
2026-10-26,91.160004
2026-10-27,91.059998
2026-10-28,91.059998
2026-10-29,91.110001


#### Federal Funds Effective Rate (DFF)

In [12]:
dff_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DFF": "federal_funds_effective_rate"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["federal_funds_effective_rate"]
dff_df["date"] = pd.to_datetime(dff_df["date"])
dff_df = dff_df.set_index("date")
dff_df = delete_column(dff_df)
dff_df.head()

,value
date,
1954-07-01,1.13
1954-07-02,1.25
1954-07-03,1.25
1954-07-04,1.25
1954-07-05,0.88


#### Market Yield on U.S. Treasury Securities at 2-Year Constant Maturity, Quoted on an Investment Basis (DGS2)

In [13]:
dgs2_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DGS2": "two_year_treasury_yield"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["two_year_treasury_yield"]
dgs2_df["date"] = pd.to_datetime(dgs2_df["date"])
dgs2_df = dgs2_df.set_index("date")
dgs2_df = delete_column(dgs2_df)
dgs2_df.head()

,value
date,
1976-06-01,7.26
1976-06-02,7.23
1976-06-03,7.22
1976-06-04,7.12
1976-06-07,7.09


#### Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis (DGS10)

In [14]:
dgs10_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DGS10": "ten_year_treasury_yield"},
    frequency=frequency,
    aggregation_method=aggregation_method,
)["ten_year_treasury_yield"]
dgs10_df["date"] = pd.to_datetime(dgs10_df["date"])
dgs10_df = dgs10_df.set_index("date")
dgs10_df = delete_column(dgs10_df)
dgs10_df.head()

,value
date,
1962-01-02,4.06
1962-01-03,4.03
1962-01-04,3.99
1962-01-05,4.02
1962-01-08,4.03


#### U.S. Dollars to Australian Dollar Spot Exchange Rate (DEXUSAL)

In [15]:
dxy_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DEXUSAL": "DXY"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["DXY"]
dxy_df["date"] = pd.to_datetime(dxy_df["date"])
dxy_df = dxy_df.set_index("date")
dxy_df = delete_column(dxy_df)
dxy_df.tail()

,value
date,
2026-10-19,0.6981
2026-10-20,0.7004
2026-10-21,0.7026
2026-10-22,0.7020
2026-10-23,0.7037


#### Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis, Inflation-Indexed (DFII10)

In [16]:
real_yield_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"DFII10": "real_10y"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["real_10y"]
real_yield_df["date"] = pd.to_datetime(real_yield_df["date"])
real_yield_df = real_yield_df.set_index("date")
real_yield_df = delete_column(real_yield_df)
real_yield_df.tail()

,value
date,
2026-10-22,2.64
2026-10-23,2.66
2026-10-26,2.66
2026-10-27,2.64
2026-10-28,2.67


#### ICE BofA US High Yield Index Option-Adjusted Spread (BAMLH0A0HYM2)

In [17]:
credit_spread_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLH0A0HYM2": "credit_spread"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["credit_spread"]
credit_spread_df["date"] = pd.to_datetime(credit_spread_df["date"])
credit_spread_df = credit_spread_df.set_index("date")
credit_spread_df = delete_column(credit_spread_df)
credit_spread_df.tail()

,value
date,
2026-10-23,3.38
2026-10-26,3.30
2026-10-27,3.18
2026-10-28,3.14
2026-10-29,3.26


#### ICE BofA AAA US Corporate Index Option-Adjusted Spread (BAMLC0A1CAAA)

In [18]:
aaa_oas_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLC0A1CAAA": "AAA_OAS"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["AAA_OAS"]
aaa_oas_df["date"] = pd.to_datetime(aaa_oas_df["date"])
aaa_oas_df = aaa_oas_df.set_index("date")
aaa_oas_df = delete_column(aaa_oas_df)
aaa_oas_df.tail()

,value
date,
2026-10-23,0.44
2026-10-26,0.43
2026-10-27,0.42
2026-10-28,0.41
2026-10-29,0.46


#### ICE BofA BBB US Corporate Index Option-Adjusted Spread (BAMLC0A4CBBB)

In [19]:
bbb_oas_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"BAMLC0A4CBBB": "BBB_OAS"},
    frequency=frequency,
    aggregation_method=aggregation_method
)["BBB_OAS"]
bbb_oas_df["date"] = pd.to_datetime(bbb_oas_df["date"])
bbb_oas_df = bbb_oas_df.set_index("date")
bbb_oas_df = delete_column(bbb_oas_df)
bbb_oas_df.tail()

,value
date,
2026-10-23,1.13
2026-10-26,1.12
2026-10-27,1.09
2026-10-28,1.07
2026-10-29,1.10


#### Federal Funds Effective Rate (FEDFUNDS)

In [20]:
ffer_df = get_fred_timeseries_data(
    observation_start=start,
    observation_end=end,
    series_mapping={"FEDFUNDS": "ff_rate"},
    frequency='m',
    aggregation_method=aggregation_method
)["ff_rate"]
ffer_df["date"] = pd.to_datetime(ffer_df["date"])
ffer_df = ffer_df.set_index("date")
ffer_df = delete_column(ffer_df)
ffer_df.tail()

,value
date,
2026-05-01,3.63
2026-06-01,3.63
2026-07-01,3.63
2026-08-01,3.63
2026-09-01,3.75


#### データ統合

In [21]:
# ============================================================
# データ統合
# ============================================================
# 各データを共通フォーマットへ変換
df_list = [
    spxew_df.rename(columns={'value': "SPXEW"}),
    sp500_df.rename(columns={'value': "GSPC"}),
    ndxe_df.rename(columns={'value': "NDXE"}),
    qqq_df.rename(columns={'value': "IXIC"}),
    hyg_df.rename(columns={'value': "HYG"}),
    ief_df.rename(columns={'value': "IEF"}),
    dff_df.rename(columns={'value': "DFF"}),
    dgs2_df.rename(columns={'value': "DGS2"}),
    dgs10_df.rename(columns={'value': "DGS10"}),
    dxy_df.rename(columns={'value': "DXY"}),
    real_yield_df.rename(columns={'value': "DFII10"}),
    credit_spread_df.rename(columns={'value': "BAMLH0A0HYM2"}),
    aaa_oas_df.rename(columns={'value': "BAMLC0A1CAAA"}),
    bbb_oas_df.rename(columns={'value': "BAMLC0A4CBBB"}),
    ffer_df.rename(columns={'value': "FEDFUNDS"}),
]
# 日付で外部結合
market_df = pd.concat(df_list, axis=1, join="outer")
# 日付順にソート
market_df = market_df.sort_index()
# 重複日付除去
market_df = market_df[~market_df.index.duplicated(keep="first")]
# 欠損補完
market_df = market_df.ffill()
# 全列欠損行は削除
market_df = market_df.dropna(how="all")
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"V1_data_integration_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = market_df.copy()
df.to_csv(output_path)
# 確認
print("=" * 60)
print("Market Data Shape")
print(market_df.shape)
print("\nColumns")
print(market_df.columns.tolist())
print("\nHead")
display(market_df.head())
print("\nTail")
display(market_df.tail())

Market Data Shape
(27573, 15)

Columns
['SPXEW', 'GSPC', 'NDXE', 'IXIC', 'HYG', 'IEF', 'DFF', 'DGS2', 'DGS10', 'DXY', 'DFII10', 'BAMLH0A0HYM2', 'BAMLC0A1CAAA', 'BAMLC0A4CBBB', 'FEDFUNDS']

Head


/tmp/ipykernel_24/4118490640.py:23: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  market_df = pd.concat(df_list, axis=1, join="outer")


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,DFII10,BAMLH0A0HYM2,BAMLC0A1CAAA,BAMLC0A4CBBB,FEDFUNDS
date,,,,,,,,,,,,,,,
1949-11-17,NaN,15.960000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-18,NaN,16.139999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-21,NaN,16.070000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-22,NaN,16.150000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1949-11-23,NaN,16.190001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tail


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,DFII10,BAMLH0A0HYM2,BAMLC0A1CAAA,BAMLC0A4CBBB,FEDFUNDS
date,,,,,,,,,,,,,,,
2026-10-25,8841.269531,7637.180176,9893.769531,25893.949219,76.889999,91.230003,3.88,4.69,5.00,0.7037,2.66,3.38,0.44,1.13,3.75
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,2.66,3.30,0.43,1.12,3.75
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,2.64,3.18,0.42,1.09,3.75
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,2.67,3.14,0.41,1.07,3.75
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.67,5.05,0.7037,2.67,3.26,0.46,1.10,3.75


#### 1. Prepare Base Dataset

Create research universe and stationary explanatory variables.


In [22]:
# 研究用データフレームの作成
research_df = market_df.copy()
research_df.index = pd.to_datetime(research_df.index)
# GSPCおよびSPXEWの目標リターンの算出
for target in ["GSPC", "SPXEW"]:
    # 対数リターンの算出
    research_df[f"{target}_LOG_RET"] = (
        np.log(research_df[target]).diff()
    )
# 各種指標の変化率の算出
research_df["DGS10_CHANGE"] = (
    research_df["DGS10"]
    .diff()
)
# DFII10の変化率の算出
research_df["DFII10_CHANGE"] = (
    research_df["DFII10"]
    .diff()
)
# DXYの変化率の算出
research_df["DXY_CHANGE"] = (
    research_df["DXY"]
    .pct_change()
)
# HY_OASの変化率の算出
research_df["HY_OAS_CHANGE"] = (
    research_df["BAMLH0A0HYM2"]
    .diff()
)
# BBB_OASの変化率の算出
research_df["BBB_OAS_CHANGE"] = (
    research_df["BAMLC0A4CBBB"]
    .diff()
)
# HYGの対数リターンの算出
research_df["HYG_LOG_RET"] = (
    np.log(
        research_df["HYG"]
    )
    .diff()
)
# IEFの対数リターンの算出
research_df["IEF_LOG_RET"] = (
    np.log(
        research_df["IEF"]
    )
    .diff()
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"1_Prepare_Base_Dataset_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = market_df.copy()
df.to_csv(output_path)
# 研究用データフレームの保存
research_df = research_df.dropna()
research_df.tail()


,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,FEDFUNDS,GSPC_LOG_RET,SPXEW_LOG_RET,DGS10_CHANGE,DFII10_CHANGE,DXY_CHANGE,HY_OAS_CHANGE,BBB_OAS_CHANGE,HYG_LOG_RET,IEF_LOG_RET
date,,,,,,,,,,,,,,,,,,,,,
2026-10-25,8841.269531,7637.180176,9893.769531,25893.949219,76.889999,91.230003,3.88,4.69,5.00,0.7037,...,3.75,0.000000,0.000000,0.00,0.00,0.0,0.00,0.00,0.000000,0.000000
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,...,3.75,0.000330,-0.007823,0.00,0.00,0.0,-0.08,-0.01,0.002468,-0.000768
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,...,3.75,0.007793,0.003445,-0.01,-0.02,0.0,-0.12,-0.03,0.002850,-0.001098
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,...,3.75,0.004676,0.000770,0.06,0.03,0.0,-0.04,-0.02,0.005290,0.000000
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.67,5.05,0.7037,...,3.75,-0.004658,-0.000346,0.00,0.00,0.0,0.12,0.03,-0.002964,0.000549


#### 2. Wavelet Filter

Create Wavelet components.

In [23]:
# GSPCおよびSPXEWのウェーブレット変換の算出
for target in ["GSPC", "SPXEW"]:
    # 対数リターンの信号を取得
    signal = research_df[f"{target}_LOG_RET"].values
    # 信号の長さを2の累乗に揃える
    usable_length = (2 ** int(np.floor(np.log2(len(signal)))))
    # 使える長さの信号を取得
    tmp_signal = signal[-usable_length:]
    # 離散ウェーブレット変換を計算
    coeffs = pywt.swt(
        tmp_signal,
        wavelet="db4",
        level=4
    )
    # 各レベルの詳細係数をデータフレームに格納
    for level_idx, (_, detail) in enumerate(coeffs, start=1):
        # レベルを計算
        level = (4 - level_idx + 1)
        # 列名を作成するためのレベル情報
        col_name = (f"{target}_WAVELET_D{level}")
        # データフレームに詳細係数を格納
        research_df.loc[research_df.index[-usable_length:], col_name] = detail
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"1_Prepare_Base_Dataset_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
research_df.tail()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,HYG_LOG_RET,IEF_LOG_RET,GSPC_WAVELET_D4,GSPC_WAVELET_D3,GSPC_WAVELET_D2,GSPC_WAVELET_D1,SPXEW_WAVELET_D4,SPXEW_WAVELET_D3,SPXEW_WAVELET_D2,SPXEW_WAVELET_D1
date,,,,,,,,,,,,,,,,,,,,,
2026-10-25,8841.269531,7637.180176,9893.769531,25893.949219,76.889999,91.230003,3.88,4.69,5.00,0.7037,...,0.000000,0.000000,-0.003940,0.001271,-0.007413,-0.000206,-0.004584,0.002752,-0.000616,-0.001611
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,...,0.002468,-0.000768,-0.001472,-0.001548,0.002466,-0.007230,-0.002305,0.004559,-0.001183,-0.001268
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,...,0.002850,-0.001098,0.001472,-0.004724,0.003231,0.005433,0.000026,0.005119,-0.000606,-0.003194
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,...,0.005290,0.000000,0.003250,-0.009015,-0.002203,0.003707,0.002341,0.001098,-0.001194,0.005381
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.67,5.05,0.7037,...,-0.002964,0.000549,0.004163,-0.009094,-0.000728,-0.005487,0.004981,-0.004010,0.000358,-0.003136


#### 3. Kalman Filter

Estimate latent trend.

In [ ]:
# 各対象の対数収益率にカルマンフィルタを適用し、平滑化された値を `research_df` に格納する。
for target in ["GSPC", "SPXEW"]:
    # 対象の対数収益率を取得し、欠損値を0で埋める
    series = (
        research_df[f"{target}_LOG_RET"]
        .fillna(0)
        .values
    )
    # カルマンフィルタを初期化する
    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        initial_state_mean=0
    )
    # カルマンフィルタを適用して平滑化された状態を取得する
    state_means, _ = (kf.smooth(series))
    # 平滑化された値を `research_df` に格納する
    research_df[f"{target}_KALMAN"] = state_means.flatten()
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"3_Kalman_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
research_df.tail()

#### 4. HP Filter

Trend and cycle decomposition.

In [26]:
# 各対象の対数収益率にHPフィルタを適用し、サイクル成分とトレンド成分を `research_df` に格納する
for target in ["GSPC", "SPXEW"]:
    # 対象の対数収益率を取得し、HPフィルタを適用する
    cycle, trend = hpfilter(
        research_df[f"{target}_LOG_RET"],
        lamb=129600
    )
    # サイクル成分とトレンド成分を `research_df` に格納する
    research_df[f"{target}_HP_CYCLE"] = cycle
    # トレンド成分を `research_df` に格納する
    research_df[f"{target}_HP_TREND"] = trend
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"4_HP_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
research_df.tail()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,SPXEW_WAVELET_D4,SPXEW_WAVELET_D3,SPXEW_WAVELET_D2,SPXEW_WAVELET_D1,GSPC_KALMAN,SPXEW_KALMAN,GSPC_HP_CYCLE,GSPC_HP_TREND,SPXEW_HP_CYCLE,SPXEW_HP_TREND
date,,,,,,,,,,,,,,,,,,,,,
2026-10-25,8841.269531,7637.180176,9893.769531,25893.949219,76.889999,91.230003,3.88,4.69,5.00,0.7037,...,-0.004584,0.002752,-0.000616,-0.001611,0.000059,-0.000421,0.000194,-0.000194,-0.001103,0.001103
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,...,-0.002305,0.004559,-0.001183,-0.001268,0.001433,-0.002611,0.000533,-0.000203,-0.008974,0.001151
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,...,0.000026,0.005119,-0.000606,-0.003194,0.003910,0.000413,0.008005,-0.000212,0.002246,0.001198
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,...,0.002341,0.001098,-0.001194,0.005381,0.002503,0.000404,0.004898,-0.000221,-0.000476,0.001246
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.67,5.05,0.7037,...,0.004981,-0.004010,0.000358,-0.003136,-0.001078,0.000029,-0.004428,-0.000230,-0.001640,0.001294


#### 5. Butterworth Filter

Long-cycle component.

In [28]:
# 各対象の対数収益率にバターワースフィルタを適用し、フィルタ後の信号を `research_df` に格納する
for target in ["GSPC", "SPXEW"]:
    # 対象の対数収益率を取得する
    signal = research_df[f"{target}_LOG_RET"].values
    # バターワースフィルタの係数を計算する
    b, a = butter(
        N=2,
        Wn=0.05,
        btype="lowpass"
    )
    # バターワースフィルタを適用して信号をフィルタリングする
    filtered = lfilter(b, a, signal)
    # フィルタ後の信号を `research_df` に格納する
    research_df[f"{target}_BUTTER"] = filtered
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"5_Butterworth_Filter_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = research_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
research_df.tail()

,SPXEW,GSPC,NDXE,IXIC,HYG,IEF,DFF,DGS2,DGS10,DXY,...,SPXEW_WAVELET_D2,SPXEW_WAVELET_D1,GSPC_KALMAN,SPXEW_KALMAN,GSPC_HP_CYCLE,GSPC_HP_TREND,SPXEW_HP_CYCLE,SPXEW_HP_TREND,GSPC_BUTTER,SPXEW_BUTTER
date,,,,,,,,,,,,,,,,,,,,,
2026-10-25,8841.269531,7637.180176,9893.769531,25893.949219,76.889999,91.230003,3.88,4.69,5.00,0.7037,...,-0.000616,-0.001611,0.000059,-0.000421,0.000194,-0.000194,-0.001103,0.001103,-0.000860,0.001923
2026-10-26,8772.370117,7639.700195,9917.309570,26026.869141,77.080002,91.160004,3.88,4.68,5.00,0.7037,...,-0.001183,-0.001268,0.001433,-0.002611,0.000533,-0.000203,-0.008974,0.001151,-0.000990,0.001968
2026-10-27,8802.639648,7699.470215,10010.910156,26286.310547,77.300003,91.059998,3.88,4.70,4.99,0.7037,...,-0.000606,-0.003194,0.003910,0.000413,0.008005,-0.000212,0.002246,0.001198,-0.001025,0.001893
2026-10-28,8809.419922,7735.560059,10064.759766,26541.609375,77.709999,91.059998,3.88,4.67,5.05,0.7037,...,-0.001194,0.005381,0.002503,0.000404,0.004898,-0.000221,-0.000476,0.001246,-0.000917,0.001790
2026-10-29,8806.370117,7699.609863,10038.530273,26201.820312,77.480003,91.110001,3.88,4.67,5.05,0.7037,...,0.000358,-0.003136,-0.001078,0.000029,-0.004428,-0.000230,-0.001640,0.001294,-0.000740,0.001693


#### 6. Create Target List

In [29]:
# 各対象のフィルタ後の列名をリスト化する
target_columns = []
for target in ["GSPC", "SPXEW"]:
    # 各対象のフィルタ後の列名をリストに追加する
    target_columns.extend([
        f"{target}_WAVELET_D1",
        f"{target}_WAVELET_D2",
        f"{target}_WAVELET_D3",
        f"{target}_WAVELET_D4",
        f"{target}_KALMAN",
        f"{target}_HP_CYCLE",
        f"{target}_HP_TREND",
        f"{target}_BUTTER",
    ])
# 作成したリストを表示する
target_columns

['GSPC_WAVELET_D1',
 'GSPC_WAVELET_D2',
 'GSPC_WAVELET_D3',
 'GSPC_WAVELET_D4',
 'GSPC_KALMAN',
 'GSPC_HP_CYCLE',
 'GSPC_HP_TREND',
 'GSPC_BUTTER',
 'SPXEW_WAVELET_D1',
 'SPXEW_WAVELET_D2',
 'SPXEW_WAVELET_D3',
 'SPXEW_WAVELET_D4',
 'SPXEW_KALMAN',
 'SPXEW_HP_CYCLE',
 'SPXEW_HP_TREND',
 'SPXEW_BUTTER']

#### 7. Pearson Correlation

In [30]:
# 各特徴量とフィルタ後の対象列のピアソン相関を計算する
features = [
    "DGS10_CHANGE",
    "DFII10_CHANGE",
    "DXY_CHANGE",
    "HY_OAS_CHANGE",
    "BBB_OAS_CHANGE",
    "HYG_LOG_RET",
    "IEF_LOG_RET",
]
# ピアソン相関の計算結果を格納するリストを初期化する
results = []
for feature in features:
    # 各特徴量とフィルタ後の対象列の組み合わせごとに相関を計算する
    for target in target_columns:
        # 対象列と特徴量のデータを抽出し、欠損値を除去する
        tmp = (research_df[[feature, target]].dropna())
        # 相関を計算するためのデータを準備する
        corr = (tmp[feature].corr(tmp[target]))
        # 計算結果をリストに追加する
        results.append({
            "feature": feature,
            "target": target,
            "pearson": corr,
        })
# 計算結果をデータフレームに変換する
pearson_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"7_Pearson_Correlation_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = pearson_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
pearson_df.sort_values("pearson",ascending=False).head(30)

,feature,target,pearson
85,HYG_LOG_RET,GSPC_HP_CYCLE,0.732792
84,HYG_LOG_RET,GSPC_KALMAN,0.589709
93,HYG_LOG_RET,SPXEW_HP_CYCLE,0.439265
92,HYG_LOG_RET,SPXEW_KALMAN,0.416469
37,DXY_CHANGE,GSPC_HP_CYCLE,0.371394
36,DXY_CHANGE,GSPC_KALMAN,0.365856
44,DXY_CHANGE,SPXEW_KALMAN,0.289065
45,DXY_CHANGE,SPXEW_HP_CYCLE,0.276734
67,BBB_OAS_CHANGE,GSPC_WAVELET_D4,0.180493
51,HY_OAS_CHANGE,GSPC_WAVELET_D4,0.172643


#### 8. Spearman Correlation

In [31]:
# 各特徴量とフィルタ後の対象列のスピアマン相関を計算する
results = []
for feature in features:
    # 各特徴量とフィルタ後の対象列の組み合わせごとに相関を計算する
    for target in target_columns:
        # 対象列と特徴量のデータを抽出し、欠損値を除去する
        tmp = (research_df[[feature, target]].dropna())
        # スピアマン相関を計算する
        coef, pvalue = spearmanr(
            tmp[feature],
            tmp[target]
        )
        # 計算結果をリストに追加する
        results.append({
            "feature": feature,
            "target": target,
            "spearman": coef,
            "pvalue": pvalue,
        })
# 計算結果をデータフレームに変換する
spearman_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"8_Spearman_Correlation_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = spearman_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
spearman_df.sort_values("spearman",ascending=False).head(30)

,feature,target,spearman,pvalue
85,HYG_LOG_RET,GSPC_HP_CYCLE,0.581478,5.807223e-100
84,HYG_LOG_RET,GSPC_KALMAN,0.473752,2.726633e-62
92,HYG_LOG_RET,SPXEW_KALMAN,0.316480,7.133641e-27
93,HYG_LOG_RET,SPXEW_HP_CYCLE,0.271983,5.240834e-20
36,DXY_CHANGE,GSPC_KALMAN,0.264997,4.888002e-19
37,DXY_CHANGE,GSPC_HP_CYCLE,0.244892,2.101558e-16
80,HYG_LOG_RET,GSPC_WAVELET_D1,0.171710,3.218266e-08
67,BBB_OAS_CHANGE,GSPC_WAVELET_D4,0.161004,2.222802e-07
44,DXY_CHANGE,SPXEW_KALMAN,0.155292,2.445006e-07
51,HY_OAS_CHANGE,GSPC_WAVELET_D4,0.142983,4.358818e-06


#### 9. Mutual Information

In [32]:
# 特徴量とターゲット列の間の相互情報を計算する
results = []
for feature in features:
    # 各特徴量に対してターゲット列との相互情報を計算する
    for target in target_columns:
        # 欠損値を除外して特徴量とターゲット列のデータを取得する
        tmp = (research_df[[feature, target]].dropna())
        # 特徴量とターゲット列を分ける
        X = tmp[[feature]]
        # ターゲット列を取得する
        y = tmp[target]
        # 相互情報を計算する
        mi = (mutual_info_regression(X, y, random_state=42)[0])
        # 計算結果を辞書に追加する
        results.append({
            "feature": feature,
            "target": target,
            "mi": mi,
        })
# 相互情報の計算結果をデータフレームに変換する
mi_df = pd.DataFrame(results)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"9_Mutual_Information_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = mi_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
# 相互情報の高い順に並べ替えて上位30件を表示する
mi_df.sort_values("mi", ascending=False).head(30)

,feature,target,mi
85,HYG_LOG_RET,GSPC_HP_CYCLE,0.440788
93,HYG_LOG_RET,SPXEW_HP_CYCLE,0.292698
109,IEF_LOG_RET,SPXEW_HP_CYCLE,0.289397
53,HY_OAS_CHANGE,GSPC_HP_CYCLE,0.287203
13,DGS10_CHANGE,SPXEW_HP_CYCLE,0.271981
45,DXY_CHANGE,SPXEW_HP_CYCLE,0.259700
37,DXY_CHANGE,GSPC_HP_CYCLE,0.256669
101,IEF_LOG_RET,GSPC_HP_CYCLE,0.245569
84,HYG_LOG_RET,GSPC_KALMAN,0.244763
21,DFII10_CHANGE,GSPC_HP_CYCLE,0.203495


#### 10. Filter Ranking Summary

Which filter consistently extracts
the strongest explanatory power?

In [33]:
# 相互情報、スピアマン相関、ピアソン相関を統合して上位50件を表示する
summary_df = (
    mi_df
    .merge(spearman_df, on=["feature", "target",])
    .merge(pearson_df, on=["feature", "target",])
    .sort_values("mi", ascending=False)
)
# 保存先のディレクトリとファイルパスを設定する
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"9_Mutual_Information_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"
# 重複するファイルがある場合は削除
if output_path.exists():
    output_path.unlink()
# 保存
df = summary_df.copy()
df.to_csv(output_path)
# 保存したデータの確認
summary_df.head(50)

,feature,target,mi,spearman,pvalue,pearson
85,HYG_LOG_RET,GSPC_HP_CYCLE,0.440788,0.581478,5.807223e-100,0.732792
93,HYG_LOG_RET,SPXEW_HP_CYCLE,0.292698,0.271983,5.240834e-20,0.439265
109,IEF_LOG_RET,SPXEW_HP_CYCLE,0.289397,0.053042,7.949501e-02,0.084417
53,HY_OAS_CHANGE,GSPC_HP_CYCLE,0.287203,-0.339093,7.668986e-31,-0.425735
13,DGS10_CHANGE,SPXEW_HP_CYCLE,0.271981,-0.034972,2.477766e-01,-0.039334
45,DXY_CHANGE,SPXEW_HP_CYCLE,0.259700,0.103228,6.272307e-04,0.276734
37,DXY_CHANGE,GSPC_HP_CYCLE,0.256669,0.244892,2.101558e-16,0.371394
101,IEF_LOG_RET,GSPC_HP_CYCLE,0.245569,0.103255,6.251553e-04,0.043312
84,HYG_LOG_RET,GSPC_KALMAN,0.244763,0.473752,2.726633e-62,0.589709
21,DFII10_CHANGE,GSPC_HP_CYCLE,0.203495,-0.056519,6.165461e-02,-0.046372
